[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snuuq/ztm-tutorial/blob/main/week4/assignment.ipynb)

> **Colab 사용 시**: 열자마자 `파일 → Drive에 사본 저장`으로 사본을 만들어 작업하세요.
> 노트북이 만드는 파일(`*.py`, `augmented_batch.png`, `runs/`, `models/`)은 세션이 초기화되기 전에 다운로드하세요.
> `.py` 파일은 노트북의 `%%writefile` 셀을 실행하면 현재 작업 폴더에 생성됩니다.

# 4주차 과제 — 이미지 폴더를 학습 스크립트로 연결하기

ants, bees 이미지 폴더를 직접 만든 Dataset으로 불러옵니다.
확인이 끝난 코드를 네 개의 스크립트로 나누고 TensorBoard에서 세 번의 실험을 비교합니다.
필요한 구현은 교재 [04장](https://www.learnpytorch.io/04_pytorch_custom_datasets/), [05장](https://www.learnpytorch.io/05_pytorch_going_modular/), [07장](https://www.learnpytorch.io/07_pytorch_experiment_tracking/)에서 확인할 수 있습니다.

## 해야 할 일
1. ants/bees 데이터를 내려받고 폴더 구조 확인
2. `Dataset`을 상속한 `ImageFolderCustom` 작성
3. `torchvision.datasets.ImageFolder`와 출력 일치 검증
4. augmentation이 적용된 한 배치 시각화
5. `data_setup.py`, `model_builder.py`, `engine.py`, `train.py` 작성
6. 명령행 인자로 세 번의 실험 실행
7. TensorBoard 곡선을 비교하고 가장 좋은 설정 설명

## 규칙
- `# TODO` 부분을 채우세요. 그 외 제공 코드는 수정하지 않아도 됩니다.
- 재현성을 위해 시드는 제공된 값(2525)을 그대로 쓰세요.
- `runs/`와 모델 `.pth` 파일은 다시 만들 수 있으므로 별도로 보관하지 않아도 됩니다.

## 0. 준비

Library import와 device 설정, 시드 고정.

In [ ]:
import os
import urllib.request
import zipfile
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from PIL import Image
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(torch.__version__, "| device:", device)

RANDOM_SEED = 2525
torch.manual_seed(RANDOM_SEED)

## 1. ants/bees 데이터 다운로드

PyTorch 튜토리얼에서 제공하는 작은 이미지 데이터입니다.
이미 내려받은 폴더가 있으면 다운로드와 압축 해제를 건너뜁니다.

In [ ]:
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
DATA_ROOT = Path("data")
ZIP_PATH = DATA_ROOT / "hymenoptera_data.zip"
DATA_PATH = DATA_ROOT / "hymenoptera_data"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATA_PATH.is_dir():
    print(f"{DATA_PATH}가 이미 있어 다운로드를 건너뜁니다.")
else:
    if not ZIP_PATH.is_file():
        print("데이터 다운로드 중...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_ROOT)

train_dir = DATA_PATH / "train"
test_dir = DATA_PATH / "val"
print("train:", train_dir)
print("test:", test_dir)

## 2. 폴더 구조 확인

각 split과 class에 이미지가 몇 장 있는지 확인합니다.

In [ ]:
for split_dir in [train_dir, test_dir]:
    print(f"\n[{split_dir.name}]")
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        image_count = len([path for path in class_dir.iterdir() if path.is_file()])
        print(f"{class_dir.name}: {image_count}")

## 3. Transform

출력 일치 검증용 transform에는 랜덤 변환을 넣지 않습니다.
학습용 transform에는 resize와 horizontal flip을 적용하세요.

In [ ]:
# TODO: Resize((64, 64))와 ToTensor()를 순서대로 적용하세요
deterministic_transform = None

# TODO: Resize((64, 64)), RandomHorizontalFlip(p=0.5), ToTensor()를 적용하세요
train_transform = None

## 4. ImageFolder 기준 결과

직접 만든 Dataset이 맞게 동작하는지 비교할 기준을 먼저 만듭니다.

In [ ]:
# TODO: train_dir과 deterministic_transform으로 ImageFolder를 만드세요
imagefolder_data = None

print("classes:", imagefolder_data.classes)
print("class_to_idx:", imagefolder_data.class_to_idx)
print("samples:", len(imagefolder_data))

reference_image, reference_label = imagefolder_data[0]
print(reference_image.shape, reference_label)

## 5. Custom Dataset 작성

`__init__`에서 이미지 경로와 class 정보를 준비하고, `__getitem__`에서 이미지와 정수 라벨을 반환하세요.
이미지는 RGB로 변환한 뒤 transform을 적용합니다.

In [ ]:
class ImageFolderCustom(Dataset):
    def __init__(self, targ_dir: str | Path, transform=None):
        # TODO: targ_dir 바로 아래 class 폴더의 .jpg/.jpeg 경로를 정렬해 저장하세요
        #       (힌트: Path(targ_dir).glob("*/*")와 path.suffix.lower()를 사용하세요)
        self.paths = None

        # TODO: class 폴더 이름을 알파벳순으로 저장하세요
        self.classes = None

        # TODO: {"ants": 0, "bees": 1} 형태의 dictionary를 만드세요
        self.class_to_idx = None
        self.transform = transform

    def __len__(self):
        # TODO: 전체 이미지 수를 반환하세요
        return None

    def __getitem__(self, index):
        # TODO: index에 해당하는 경로를 고르세요
        image_path = None

        # 파일 핸들이 남지 않게 with 문 안에서 열고 RGB 이미지로 복사합니다.
        with Image.open(image_path) as image:
            image = image.convert("RGB")

        # TODO: 부모 폴더 이름으로 정수 라벨을 찾으세요
        label = None

        if self.transform is not None:
            image = self.transform(image)

        return image, label

## 6. ImageFolder와 출력 일치 검증

랜덤 변환이 없는 같은 transform을 사용해 길이, class 정보, 첫 10개 이미지와 라벨을 비교합니다.
모든 assertion이 통과하면 구현이 맞습니다.

In [ ]:
custom_data = ImageFolderCustom(
    targ_dir=train_dir,
    transform=deterministic_transform,
)

assert len(custom_data) == len(imagefolder_data)
assert custom_data.classes == imagefolder_data.classes
assert custom_data.class_to_idx == imagefolder_data.class_to_idx

for index in range(10):
    custom_image, custom_label = custom_data[index]
    reference_image, reference_label = imagefolder_data[index]
    assert custom_label == reference_label
    assert torch.allclose(custom_image, reference_image)

print("첫 10개 샘플의 이미지와 라벨이 모두 같습니다.")

## 7. Augmentation batch 저장

학습용 transform을 적용한 Custom Dataset과 DataLoader를 만들고 첫 배치를 저장합니다.
이미지 위에는 class 이름을 표시하세요.

In [ ]:
# TODO: train_transform을 사용하는 Custom Dataset을 만드세요
augmented_data = None

# TODO: batch_size=8, shuffle=True, num_workers=0인 DataLoader를 만드세요
augmented_dataloader = None

# TODO: 첫 batch를 꺼내세요
images, labels = None, None

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for index, ax in enumerate(axes.flat):
    ax.imshow(images[index].permute(1, 2, 0))
    ax.set_title(augmented_data.classes[int(labels[index])])
    ax.axis(False)

plt.tight_layout()
plt.savefig("augmented_batch.png", dpi=150, bbox_inches="tight")
print("그래프 저장: augmented_batch.png")

## 8. 노트북 코드를 네 파일로 나누기

이제 학습 코드를 다음 구조로 나눕니다.

```text
data_setup.py      # Dataset과 DataLoader
model_builder.py   # TinyVGG
engine.py          # train_step, test_step, train
train.py           # argparse와 전체 실행
```

아래 `%%writefile` 셀을 실행하면 현재 작업 폴더에 실제 파일이 만들어집니다.

## 9. `data_setup.py`

In [ ]:
%%writefile data_setup.py
import os
from torch.utils.data import DataLoader
from torchvision import datasets


def create_dataloaders(
    train_dir,
    test_dir,
    train_transform,
    test_transform,
    batch_size,
    num_workers=0,
):
    # TODO: ImageFolder로 train/test Dataset을 만드세요
    train_data = None
    test_data = None

    class_names = train_data.classes

    # TODO: train은 shuffle=True, test는 shuffle=False인 DataLoader를 만드세요
    train_dataloader = None
    test_dataloader = None

    return train_dataloader, test_dataloader, class_names

## 10. `model_builder.py`

In [ ]:
%%writefile model_builder.py
import torch
from torch import nn


class TinyVGG(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()

        # TODO: Conv2d → ReLU → Conv2d → ReLU → MaxPool2d block 두 개를 만드세요
        self.conv_block_1 = None
        self.conv_block_2 = None

        # 입력 이미지는 64×64이고 pooling을 두 번 거치면 16×16이 됩니다.
        # TODO: Flatten과 Linear로 output_shape개의 logits를 출력하세요
        self.classifier = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: 두 convolution block과 classifier를 순서대로 통과시키세요
        return None

## 11. `engine.py`

TensorBoard 기록을 위해 `train` 함수가 `writer`를 선택적으로 받게 합니다.

In [ ]:
%%writefile engine.py
import torch
from torch import nn


def train_step(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        # TODO: forward와 loss
        logits = None
        loss = None

        # TODO: zero_grad → backward → step



        # TODO: 샘플 수를 반영해 loss와 맞힌 개수를 누적하세요


    # TODO: 전체 샘플 기준 평균 loss와 accuracy(0~1)를 반환하세요
    return None, None


def test_step(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            # TODO: logits, loss, 예측 라벨을 계산하고 결과를 누적하세요



    # TODO: 전체 샘플 기준 평균 loss와 accuracy(0~1)를 반환하세요
    return None, None


def train(
    model,
    train_dataloader,
    test_dataloader,
    optimizer,
    loss_fn,
    epochs,
    device,
    writer=None,
):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
    }

    for epoch in range(epochs):
        # TODO: train_step과 test_step을 호출하세요
        train_loss, train_acc = None, None
        test_loss, test_acc = None, None

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if writer is not None:
            # TODO: add_scalars로 Loss/train·test와 Accuracy/train·test를 기록하세요
            pass

        print(
            f"Epoch: {epoch + 1:2d} | "
            f"Train loss: {train_loss:.4f}, acc: {train_acc:.3f} | "
            f"Test loss: {test_loss:.4f}, acc: {test_acc:.3f}"
        )

    if writer is not None:
        writer.close()

    return results

## 12. `train.py`

`argparse`로 경로와 주요 하이퍼파라미터를 받습니다.
각 실행은 `run_name_실행시각` 폴더에 TensorBoard 로그를 남깁니다.

In [ ]:
%%writefile train.py
import argparse
from datetime import datetime
from pathlib import Path

import torch
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

import data_setup
import engine
import model_builder


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-dir", default="data/hymenoptera_data/train")
    parser.add_argument("--test-dir", default="data/hymenoptera_data/val")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--hidden-units", type=int, default=16)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--run-name", default="baseline")
    parser.add_argument("--log-dir", default="runs")
    return parser.parse_args()


def main():
    args = parse_args()
    torch.manual_seed(2525)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # TODO: train에는 Resize, RandomHorizontalFlip, ToTensor를 적용하세요
    #       test에는 Resize와 ToTensor만 적용하세요. 크기는 모두 64×64입니다.
    train_transform = None
    test_transform = None

    # TODO: data_setup.create_dataloaders를 호출하세요
    train_dataloader, test_dataloader, class_names = None, None, None

    # TODO: RGB 입력, args.hidden_units, len(class_names)를 사용하는 TinyVGG를 만드세요
    model = None

    # TODO: CrossEntropyLoss와 Adam optimizer를 만드세요
    loss_fn = None
    optimizer = None

    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(args.log_dir) / f"{args.run_name}_{timestamp}"

    # TODO: run_dir을 사용하는 SummaryWriter를 만드세요
    writer = None
    print(f"TensorBoard log: {run_dir}")

    results = engine.train(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        epochs=args.epochs,
        device=device,
        writer=writer,
    )

    model_dir = Path("models")
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{args.run_name}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"모델 저장: {model_path}")


if __name__ == "__main__":
    main()

## 13. 스크립트 문법 검사

네 파일이 Python 문법에 맞는지 먼저 확인합니다.
오류가 나오면 해당 파일을 만드는 `%%writefile` 셀로 돌아가 수정하세요.

In [ ]:
!python -m py_compile data_setup.py model_builder.py engine.py train.py

## 14. 1-epoch smoke test

전체 연결이 되는지 1 epoch만 먼저 실행합니다.
세 실험의 로그와 섞이지 않게 smoke test 로그는 `runs_smoke/`에 저장합니다.

In [ ]:
!python train.py --epochs 1 --lr 0.001 --run-name smoke --log-dir runs_smoke

## 15. 세 번의 실험

한 번에 한 조건만 바꿉니다.
baseline과 high_lr는 learning rate, baseline과 longer는 epoch 수의 영향을 비교합니다.

In [ ]:
!python train.py --epochs 5 --lr 0.001 --run-name baseline --log-dir runs
!python train.py --epochs 5 --lr 0.01 --run-name high_lr --log-dir runs
!python train.py --epochs 10 --lr 0.001 --run-name longer --log-dir runs

## 16. TensorBoard

세 실험의 Loss와 Accuracy를 같은 화면에서 비교하세요.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 17. 실험 비교

아래 질문에 답하세요.

1. test accuracy가 가장 높은 실험은 무엇인가요?
2. test loss가 다시 증가하거나 불안정한 실험이 있나요?
3. 다음 실험에서 값 하나만 바꾼다면 무엇을 어떻게 바꾸겠나요?

> 답:

## 18. (선택 도전) `predict.py`

저장된 모델 경로와 이미지 경로를 명령행 인자로 받아 ants/bees 예측을 출력하는 `predict.py`를 작성하세요.

```bash
python predict.py --model-path models/baseline.pth --image-path data/hymenoptera_data/val/ants/170652283_ecdaff5d1a.jpg
```

In [ ]:
# (선택) 자유롭게 작성

---

## 체크리스트

- [ ] 모든 셀이 위에서 아래로 에러 없이 실행됨
- [ ] Custom Dataset과 ImageFolder의 길이, class 정보가 같음
- [ ] deterministic transform에서 첫 10개 이미지가 `torch.allclose`
- [ ] `augmented_batch.png`가 생성됨
- [ ] 4개의 `.py` 파일이 `py_compile`을 통과함
- [ ] 1-epoch smoke test가 정상 종료됨
- [ ] TensorBoard에 baseline, high_lr, longer 실험이 표시됨
- [ ] 가장 좋은 설정과 판단 근거를 작성함